# 📘 Notebook 7 — Riduzione della dimensionalità: PCA

Quando i dati hanno **molte caratteristiche** (es: 100, 1000, 10000) sorgono problemi:
- I modelli diventano lenti
- Si fa **overfitting** più facilmente (curse of dimensionality)
- Diventa impossibile **visualizzarli**

La **riduzione della dimensionalità** ci permette di **comprimere** i dati mantenendo l'informazione importante.

## 🎯 Cosa imparerai
1. Cos'è e perché serve la riduzione di dimensionalità
2. L'idea della **PCA** (Principal Component Analysis)
3. Come **visualizzare** dataset complessi in 2D
4. Come usare la PCA come **preprocessing** prima di altri modelli
5. Cenni a t-SNE e UMAP

## 🤔 Quando usarla?
- **Visualizzazione**: per vedere dati ad alta dimensionalità
- **Preprocessing**: per accelerare modelli e ridurre overfitting
- **Compressione**: per archiviare dati in modo più efficiente
- **Rimozione del rumore**: PCA conserva i pattern "forti" e scarta quelli "deboli"


![PCA](Media/07/PCA.png)


## 1️⃣ L'intuizione: trovare la "vista migliore"

Immagina dei punti sparsi in 3D: la PCA cerca **gli assi (direzioni) lungo cui i punti variano di più**, e ci permette di proiettare i dati su quegli assi.

**Analogia**: se fotografi un palazzo, ci sono molte angolazioni. Quella **più informativa** è quella frontale. La PCA trova automaticamente la "foto migliore".

![dimensionatlià](Media/07/dimensionalità.png)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Generiamo dati 2D con un trend obliquo
np.random.seed(42)
n = 200
x1 = np.random.normal(0, 2, n)
x2 = 0.7 * x1 + np.random.normal(0, 0.5, n)   # x2 è correlato a x1
X = np.column_stack([x1, x2])

# Visualizziamo i dati
plt.figure(figsize=(7, 7))
plt.scatter(X[:, 0], X[:, 1], alpha=0.6, s=30)
plt.axhline(0, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Dati con direzione di massima variazione "diagonale"')
plt.grid(alpha=0.3)
plt.axis('equal')
plt.show()

# 💡 I punti variano molto in direzione obliqua e poco in direzione perpendicolare.
# La PCA scoprirà queste direzioni!

In [ ]:
from sklearn.decomposition import PCA

# Applichiamo PCA: chiediamo le 2 componenti principali
pca = PCA(n_components=2)
pca.fit(X)

# Le 'direzioni' (componenti principali) trovate
print("Componenti principali (gli 'assi' migliori):")
print(pca.components_)

# Quanta varianza spiega ogni componente
print("\nVarianza spiegata da ogni componente (in %):")
print((pca.explained_variance_ratio_ * 100).round(2))

# Visualizziamo le direzioni
plt.figure(figsize=(7, 7))
plt.scatter(X[:, 0], X[:, 1], alpha=0.4, s=30)

# Disegniamo le direzioni (frecce) lunghe quanto la dev std lungo quella direzione
for i, (componente, var) in enumerate(zip(pca.components_, pca.explained_variance_)):
    v = componente * np.sqrt(var) * 3   # scaliamo per visualizzazione
    plt.arrow(pca.mean_[0], pca.mean_[1], v[0], v[1],
              head_width=0.3, head_length=0.3,
              fc=['red', 'green'][i], ec=['red', 'green'][i],
              linewidth=2, label=f'PC{i+1}')

plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Le componenti principali trovate dalla PCA')
plt.legend()
plt.grid(alpha=0.3)
plt.axis('equal')
plt.show()

# 💡 PC1 (rossa) è la direzione di MASSIMA varianza
# PC2 (verde) è perpendicolare e cattura la varianza residua

### Cosa significa "ridurre dimensionalità"?

**Proiettare** i dati sulle componenti principali. Se teniamo solo PC1, da 2D passiamo a 1D, perdendo solo poco di informazione.

In [ ]:
# Riduciamo da 2D a 1D
pca_1d = PCA(n_components=1)
X_1d = pca_1d.fit_transform(X)

print(f"Forma originale: {X.shape}")
print(f"Forma ridotta:   {X_1d.shape}")
print(f"Varianza mantenuta: {pca_1d.explained_variance_ratio_[0] * 100:.1f}%")

# Visualizziamo i dati proiettati su 1D
plt.figure(figsize=(10, 3))
plt.scatter(X_1d, np.zeros_like(X_1d), alpha=0.5, s=30)
plt.axhline(0, color='gray', linewidth=0.5)
plt.title(f'Dati proiettati su 1 dimensione (varianza mantenuta: {pca_1d.explained_variance_ratio_[0]*100:.1f}%)')
plt.yticks([])
plt.grid(alpha=0.3, axis='x')
plt.show()

## 2️⃣ Caso reale: il dataset Digits (8x8 = 64 dimensioni!)

Carichiamo immagini 8x8 di cifre scritte a mano. Ogni immagine ha **64 valori** (i pixel). Impossibile da visualizzare direttamente. Usiamo PCA per ridurre a 2D!


![PC1](Media/07/PC1.png)


In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X, y = digits.data, digits.target
print(f"Dataset: {X.shape[0]} immagini, ognuna con {X.shape[1]} pixel")
print(f"Classi (cifre): {np.unique(y)}")

# Mostriamo qualche immagine
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f'Cifra: {y[i]}')
    ax.axis('off')
plt.suptitle('Esempi del dataset Digits')
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

# Standardizziamo i dati (PCA è sensibile alle scale!)
X_scaled = StandardScaler().fit_transform(X)

# Riduciamo da 64 a 2 dimensioni
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)

# Quanta informazione abbiamo perso?
print(f"Varianza spiegata dalle 2 componenti: {pca.explained_variance_ratio_.sum()*100:.1f}%")

# Visualizziamo!
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap='tab10', s=15, alpha=0.7)
plt.colorbar(scatter, label='Cifra')
plt.xlabel('Componente principale 1')
plt.ylabel('Componente principale 2')
plt.title('Dataset Digits ridotto da 64 a 2 dimensioni')
plt.grid(alpha=0.3)
plt.show()

# 💡 Magia: vediamo cifre simili raggruppate vicine, anche partendo da 64 dimensioni!
# Le cifre 0 e 1 sono molto separate (diverse). Le 4 e 9 si sovrappongono (somiglianti).

## 3️⃣ Quante componenti tenere?

Spesso non vogliamo solo 2 componenti. Domanda: quante ne servono per **conservare il 95% dell'informazione**?

![PC4](Media/07/PCA4.png)


In [ ]:
# PCA con TUTTE le componenti per vedere come si distribuisce la varianza
pca_full = PCA().fit(X_scaled)
#Qui dici alla PCA di analizzare il dataset originale senza porre limiti. Se il tuo dataset ha 100 colonne, la PCA calcolerà tutte e 100 le possibili componenti (PC1, PC2, PC3... fino a PC100).
varianza_cumulata = np.cumsum(pca_full.explained_variance_ratio_)
#Questo comando restituisce la percentuale di informazione contenuta in ogni singola componente. Ad esempio: la PC1 da sola ha il 60% dell'informazione, la PC2 il 20%, la PC3 il 10%, e così via a scendere.

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(varianza_cumulata)+1), varianza_cumulata, 'o-', markersize=4)
plt.axhline(0.95, color='red', linestyle='--', label='95% di varianza')
plt.xlabel('Numero di componenti')
plt.ylabel('Varianza cumulata')
plt.title('Quante componenti per conservare l\'informazione?')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Quante componenti per arrivare al 95%?
n_componenti = np.argmax(varianza_cumulata >= 0.95) + 1
#Invece di guardare il grafico a occhio, questa riga di codice interroga la lista della varianza cumulata e chiede: "Qual è il primissimo momento in cui la somma delle informazioni supera o è uguale al 95%?" La risposta viene salvata in n_componenti
print(f"Servono {n_componenti} componenti per conservare il 95% (su {X.shape[1]} originali)")
print(f"Riduzione: {(1 - n_componenti/X.shape[1])*100:.1f}% in meno!")

## 4️⃣ PCA come PREPROCESSING

Usare PCA prima di un modello può:
- velocizzare l'addestramento
- ridurre overfitting (meno variabili = meno rumore)
- talvolta migliorare le performance

![PCA2](Media/07/PCA2.png)


In [ ]:
import time
from sklearn.decomposition import PCA  # 💡 Aggiunto: fondamentale per far girare la seconda parte!
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# -------------------------------------------------------------------------
# SUDDIVISIONE DEI DATI
# -------------------------------------------------------------------------
# Dividiamo il dataset in Training Set (70%) e Test Set (30%).
# - stratify=y: assicura che la proporzione delle classi rimanga identica sia nel train che nel test (cruciale per dataset sbilanciati).
# - random_state=42: fissa il "seed" del generatore casuale, rendendo lo split riproducibile ad ogni esecuzione.
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)

# =========================================================================
# 1. MODELLO SENZA PCA (Baseline su tutte le 64 caratteristiche originali)
# =========================================================================

# Fissiamo il timestamp iniziale per calcolare il tempo di addestramento
t0 = time.time()

# Inizializziamo e addestriamo la Logistic Regression sulle 64 feature.
# - max_iter=2000: aumenta il numero di iterazioni massime rispetto al default (100)
#   per evitare che l'algoritmo non converga a causa del numero di feature o della scala dei dati.
modello1 = LogisticRegression(max_iter=2000).fit(X_train, y_train)

# Calcoliamo il tempo effettivo impiegato per il fit (tempo finale - tempo iniziale)
t1 = time.time() - t0

# Valutiamo l'accuratezza predittiva del primo modello usando il Test Set
acc1 = accuracy_score(y_test, modello1.predict(X_test))


# =========================================================================
# 2. MODELLO CON PCA (Riduzione della dimensionalità a 20 componenti)
# =========================================================================

# Inizializziamo l'algoritmo PCA impostando il target a 20 componenti principali (rispetto alle 64 iniziali)
pca = PCA(n_components=20)

# Calcoliamo la PCA sul Train Set e trasformiamo le feature originali nelle 20 nuove componenti
X_train_pca = pca.fit_transform(X_train)

# Applichiamo la STESSA trasformazione geometrica al Test Set.
# ⚠️ IMPORTANTE: Usiamo solo .transform() senza .fit() per evitare Data Leakage (contaminazione dei dati).
X_test_pca = pca.transform(X_test)

# Facciamo ripartire il timer per il secondo modello
t0 = time.time()

# Addestriamo un nuovo modello di Logistic Regression, ma stavolta usando solo le 20 feature della PCA
modello2 = LogisticRegression(max_iter=2000).fit(X_train_pca, y_train)

# Calcoliamo il tempo impiegato per l'addestramento sui dati ridotti
t2 = time.time() - t0

# Valutiamo l'accuratezza del secondo modello sul Test Set trasformato dalla PCA
acc2 = accuracy_score(y_test, modello2.predict(X_test_pca))


# =========================================================================
# 3. STAMPA E CONFRONTO DEI RISULTATI
# =========================================================================

# Mostriamo l'accuratezza e la velocità del modello sulle 64 feature originarie
print(f"Senza PCA (64 feat.): accuracy={acc1:.3f}, tempo={t1:.3f}s")

# Mostriamo l'accuratezza e la velocità del modello sulle 20 componenti PCA
print(f"Con PCA (20 feat.):   accuracy={acc2:.3f}, tempo={t2:.3f}s")

# pca.explained_variance_ratio_ restituisce la percentuale di informazione (varianza) contenuta in ogni componente.
# Sommandole (.sum()), scopriamo quanta informazione totale del dataset originale abbiamo salvato.
print(f"Varianza conservata:  {pca.explained_variance_ratio_.sum()*100:.1f}%")

# 💡 Spesso: meno features = simile accuracy + più velocità + meno overfitting

## 5️⃣ Visualizzare la "compressione" su immagini

Vediamo quanto si perde se rappresentiamo le cifre con poche componenti e poi le "ricostruiamo".

In [ ]:
# Confronto: ricostruzione con varie n_components
fig, axes = plt.subplots(4, 5, figsize=(12, 9))
indici = np.random.choice(len(X), 5, replace=False)
n_comps = [2, 8, 20, 64]

for r, n_comp in enumerate(n_comps):
    pca_r = PCA(n_components=n_comp).fit(X_scaled)
    X_compresso = pca_r.transform(X_scaled[indici])
    X_ricostruito = pca_r.inverse_transform(X_compresso)

    for c in range(5):
        ax = axes[r, c]
        # Riportiamo a forma 8x8 (e re-invertiamo lo scaling per visualizzare)
        img = X_ricostruito[c].reshape(8, 8)
        ax.imshow(img, cmap='gray')
        ax.axis('off')
        if c == 0:
            varianza = pca_r.explained_variance_ratio_.sum() * 100
            ax.set_ylabel(f'{n_comp} comp.\n({varianza:.0f}%)', rotation=0, labelpad=40, va='center')

plt.suptitle('Ricostruzione delle cifre con un numero crescente di componenti', fontsize=13)
plt.tight_layout(); plt.show()

# 💡 Con appena 8 componenti su 64 le cifre sono già riconoscibili!

## 6️⃣ Quando la PCA NON basta: t-SNE e UMAP

La PCA è **lineare**. Per dati con strutture **non lineari** complesse esistono tecniche più sofisticate:
- **t-SNE**: ottima per visualizzazione, conserva le relazioni locali
- **UMAP**: simile a t-SNE, più veloce e conserva anche struttura globale



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA  # 💡 Aggiunto: necessario per il confronto
from sklearn.manifold import TSNE

# =========================================================================
# 1. CAMPIONAMENTO DEI DATI (Dowksampling)
# =========================================================================
# t-SNE ha una complessità computazionale elevata (quadratica rispetto al numero di campioni).
# Per evitare lunghe attese, estraiamo un sottoinsieme casuale di 1000 indici senza ripetizioni.
idx = np.random.choice(len(X_scaled), 1000, replace=False)

# Selezioniamo solo i 1000 campioni estratti sia per le feature (X) che per i target (y)
X_sub = X_scaled[idx]
y_sub = y[idx]


# =========================================================================
# 2. APPLICAZIONE DI T-SNE (Non Lineare)
# =========================================================================
# Inizializziamo il t-SNE per proiettare i dati in uno spazio a 2 dimensioni.
# - perplexity=30: definisce il numero di "vicini di casa" da considerare per ogni punto.
#   Bilancia l'attenzione del modello tra la struttura locale e globale dei dati.
# - random_state=42: rende l'algoritmo riproducibile (t-SNE è probabilistico e non deterministico).
tsne = TSNE(n_components=2, random_state=42, perplexity=30)

# Calcola la proiezione bidimensionale ottimizzando la distanza tra i punti
X_tsne = tsne.fit_transform(X_sub)


# =========================================================================
# 3. APPLICAZIONE DI PCA (Lineare) PER IL CONFRONTO
# =========================================================================
# Riduciamo gli stessi 1000 campioni a 2 dimensioni usando la PCA standard.
# La PCA cerca le direzioni di massima varianza tramite combinazioni lineari delle feature originali.
X_pca = PCA(n_components=2).fit_transform(X_sub)


# =========================================================================
# 4. VISUALIZZAZIONE GRAFICA (PLOT)
# =========================================================================
# Creiamo una figura con 1 riga e 2 colonne per affiancare i grafici
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Grafico 1: Risultato PCA ---
# Disegniamo il grafico a dispersione (scatter plot).
# - c=y_sub: colora i punti in base alla cifra reale (0-9).
# - cmap='tab10': usa una palette di 10 colori netti e distinti.
# - alpha=0.7: rende i punti leggermente trasparenti per vedere le sovrapposizioni.
axes[0].scatter(
    X_pca[:, 0], X_pca[:, 1], c=y_sub, cmap="tab10", s=15, alpha=0.7
)
axes[0].set_title("PCA (lineare)")
axes[0].grid(alpha=0.3)  # Aggiunge una griglia leggera sullo sfondo

# --- Grafico 2: Risultato t-SNE ---
# Facciamo lo stesso scatter plot, ma usando le coordinate calcolate dal t-SNE.
axes[1].scatter(
    X_tsne[:, 0], X_tsne[:, 1], c=y_sub, cmap="tab10", s=15, alpha=0.7
)
axes[1].set_title("t-SNE (non lineare)")
axes[1].grid(alpha=0.3)

# Mostra la finestra con i due grafici affiancati
plt.show()

# 💡 t-SNE separa molto meglio le 10 classi!
# Però è solo per VISUALIZZARE: t-SNE non si usa per preprocessing prima di altri modelli.

## 🎓 Riepilogo del Notebook 7

### PCA
- ✅ Veloce, lineare, deterministica
- ✅ Buona per: preprocessing, compressione, visualizzazione
- ❌ Non cattura strutture non lineari
- ❌ Le componenti possono essere difficili da interpretare

### Quando usarla
- Tante variabili numeriche correlate tra loro
- Preprocessing prima di modelli sensibili al numero di feature (es: KNN)
- Visualizzazione esplorativa rapida

![conclusioni](Media/07/conclusioni.png)

### Punti chiave
| Concetto | Significato |
|---|---|
| Componente principale | Una nuova "direzione" combinazione delle originali |
| Varianza spiegata | Quanta informazione cattura ogni componente |
| Standardizzazione | OBBLIGATORIA prima della PCA |
| Inverse transform | Per "ricostruire" i dati originali |

### Riepilogo: tipi di apprendimento incontrati finora
1. **Supervisionato** (notebook 3, 4, 5):
   - Regressione: prevedere numeri
   - Classificazione: prevedere categorie
2. **Non supervisionato** (notebook 6, 7):
   - Clustering: scoprire gruppi
   - Riduzione dimensionalità: comprimere informazione

👉 **Prossimo notebook (08)**: tutto sulla **valutazione e ottimizzazione dei modelli** — cross-validation, ricerca degli iperparametri, evitare l'overfitting, e una **roadmap pratica** per affrontare un nuovo problema di ML da zero.